# Experiment 1: inference and evaluation on T2I-VeRW val

Loads a trained text-guided deformable checkpoint and scores every (query caption, gallery image) pair of the
T2I-VeRW `val` split (4,404 × 4,404 = 19.4M pairs), then reports R1/R5/R10/R50, mAP and mINP. Protocol and model
are described in `experiments/exp1/README.md`; the scoring is the same as `evaluate_pairwise` in `train.py`.

- The full evaluation takes **~23 min on one MI300X** (CLIP encoding ~1 min, pairwise scoring the rest). The score
  matrix is cached next to the checkpoint, so re-running the notebook afterwards is instant.
- `model_best.pth` is the epoch-80 checkpoint of `logs/exp1_baseline`; training logged **R1 47.03, mAP 26.85, mINP 6.54**
  for it. Expect the same numbers up to bf16 noise (±0.05).
- Set `SUBSET_PIDS` (e.g. 20) for a quick run on the first N test identities; those metrics are not comparable to the paper.

In [ ]:
REPO = '/mnt/data/ishant/T2I-ReID'
CONFIG = f'{REPO}/configs/exp1_baseline.yaml'
CKPT = f'{REPO}/logs/exp1_baseline/model_best.pth'     # or checkpoint_latest.pth / model_final.pth
DEVICE = 'cuda'
PAIRS_PER_CHUNK = 8192                                   # pairs scored per forward; 8192 uses ~33 GB
SUBSET_PIDS = None                                       # None = full val split; N = first N test identities only

In [ ]:
import os, sys, textwrap
sys.path.insert(0, REPO)

import clip
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from omegaconf import OmegaConf
from PIL import Image
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from t2i_reid.data import T2IVeRW, TextDataset, ImageDataset, build_transform
from t2i_reid.utils import t2i_metrics
from train import build_model

cfg = OmegaConf.load(CONFIG)
cfg.device = DEVICE
device = torch.device(DEVICE)
autocast = lambda: torch.autocast(device.type, dtype=torch.bfloat16, enabled=device.type == 'cuda')

dataset = T2IVeRW(root=cfg.data.root, ann_file=cfg.data.ann_file, test_split=cfg.data.test_split)
query, gallery = dataset.query, dataset.gallery              # [(pid, caption)], [(pid, image_path)]
if SUBSET_PIDS:
    keep = set(sorted({pid for pid, _ in gallery})[:SUBSET_PIDS])
    query, gallery = [q for q in query if q[0] in keep], [g for g in gallery if g[0] in keep]
q_pids = torch.tensor([pid for pid, _ in query])
g_pids = torch.tensor([pid for pid, _ in gallery])
print(f'{len(query)} queries x {len(gallery)} gallery images, {len(set(g_pids.tolist()))} identities')

model = build_model(cfg, dataset.num_train_pids)
state = torch.load(CKPT, map_location='cpu', weights_only=False)
model.load_state_dict(state.get('model', state))              # checkpoint_latest.pth keeps the weights under 'model'
model.to(device).eval()
transform = build_transform(tuple(cfg.data.image_size))
print(f'loaded {CKPT}, gamma = {model.gamma.item():.4f}')

## Score every pair

CLIP runs once per caption and once per image. Steps 2–7 (text enrichment, spatial tokens, offsets, deformable
sampling, pooling, cosine) depend on both the caption and the image, so they run on all Q × G pairs.
`dual` is the plain CLIP ranking of the same fine-tuned backbone (caption EOT vs image CLS), i.e. the model with
the deformable branch switched off.

In [ ]:
@torch.no_grad()
def score_all(model, query, gallery):
    tokenize = lambda caption: clip.tokenize(caption, truncate=True)[0]
    loader = lambda ds: DataLoader(ds, batch_size=cfg.data.test_batch_size, num_workers=8)
    with autocast():
        text = [model.encode_text(b['caption'].to(device)) for b in loader(TextDataset(query, tokenize))]
        text = tuple(torch.cat(x) for x in zip(*text))                                  # tokens, mask, eot
        image_tokens = torch.cat([model.encode_image(b['image'].to(device))
                                  for b in tqdm(loader(ImageDataset(gallery, transform)), desc='encode gallery')])
        values = model.deform.project_values(image_tokens[:, 1:])                       # once per gallery image
        num_q, num_g = len(query), len(gallery)
        sim = torch.empty(num_q * num_g, device=device)
        for start in tqdm(range(0, num_q * num_g, PAIRS_PER_CHUNK), desc='score pairs'):
            k = torch.arange(start, min(start + PAIRS_PER_CHUNK, num_q * num_g), device=device)
            sim[start:start + len(k)] = model.score_pairs(text, image_tokens, k // num_g, k % num_g, values)['score'].float()
    dual = F.normalize(text[2].float(), dim=-1) @ F.normalize(image_tokens[:, 0].float(), dim=-1).t()
    return sim.view(num_q, num_g).cpu(), dual.cpu()


cache = None if SUBSET_PIDS else CKPT.replace('.pth', f'.{int(os.path.getmtime(CKPT))}.val_sim.pt')
if cache and os.path.exists(cache):
    sim, dual = torch.load(cache).values()
    print(f'loaded cached scores from {cache}')
else:
    sim, dual = score_all(model, query, gallery)
    if cache:
        torch.save({'sim': sim, 'dual': dual}, cache)

In [ ]:
results = {'Exp1: text-guided deformable': t2i_metrics(sim, q_pids, g_pids),
           'Exp1 backbone only (CLS vs EOT)': t2i_metrics(dual, q_pids, g_pids)}
paper = {'IRRA (T2I-VeRW Table IV)': {'R1': 47.3, 'mAP': 22.8},
         'PFCVR (T2I-VeRW Table IV)': {'R1': 55.2, 'mAP': 26.2}}

cols = ['R1', 'R5', 'R10', 'R50', 'mAP', 'mINP']
print(f"{'':34s}" + ''.join(f'{c:>8s}' for c in cols))
for name, m in {**results, **paper}.items():
    print(f'{name:34s}' + ''.join(f'{m[c]:8.2f}' if c in m else f"{'-':>8s}" for c in cols))

## Retrievals

Top-5 gallery images for one query that the model gets right at rank 1 and one it gets wrong.
Green border = same vehicle as the query, red = different vehicle; the title is the score.

In [ ]:
def show_query(i, k=5):
    order = sim[i].argsort(descending=True)[:k].tolist()
    fig, axes = plt.subplots(1, k, figsize=(3 * k, 3.6))
    for rank, (ax, j) in enumerate(zip(axes, order), 1):
        pid, path = gallery[j]
        ax.imshow(Image.open(path).convert('RGB').resize((224, 224)))
        ax.set_xticks([]), ax.set_yticks([])
        ax.set_title(f'#{rank}  {sim[i, j]:.3f}', fontsize=9)
        for spine in ax.spines.values():
            spine.set_edgecolor('tab:green' if pid == query[i][0] else 'tab:red'), spine.set_linewidth(4)
    fig.suptitle(f'query {i} (pid {query[i][0]}): ' + textwrap.fill(query[i][1], 150), fontsize=9, ha='left', x=0.01)
    plt.tight_layout(), plt.show()


top1_correct = g_pids[sim.argmax(dim=1)] == q_pids
for hits in (top1_correct, ~top1_correct):                   # first success, first failure (if any)
    if hits.any():
        show_query(int(hits.nonzero()[0]))

## Where does the caption make the model look?

Deformable sampling locations for one gallery image under two captions: its own (matching) caption and a caption of a
different vehicle. Each heatmap is the attention-weighted density of all 196 queries × 8 heads × 4 points. If text
steers the geometry, the two maps should differ; the third panel is their difference (red = more mass under the
matching caption).

In [ ]:
@torch.no_grad()
def sampling_density(caption, image_path, bins=14):                  # one bin per patch
    with autocast():
        tokens, mask, _ = model.encode_text(clip.tokenize([caption], truncate=True).to(device))
        V = model.encode_image(transform(Image.open(image_path).convert('RGB'))[None].to(device))
        locations, weights = model.offsets(tokens, mask, V)                            # [1,196,H,P,2], [1,196,H,P]
    xy, w = locations.float().reshape(-1, 2).cpu().numpy(), weights.float().reshape(-1).cpu().numpy()
    density, _, _ = np.histogram2d(xy[:, 0], xy[:, 1], bins=bins, range=[[0, 1], [0, 1]], weights=w)
    return density.T / w.sum()                                                          # rows = y, cols = x


j = int(top1_correct.nonzero()[0])                     # a query the model gets right; use its top-1 image
j_img = int(sim[j].argmax())
other = int((q_pids != gallery[j_img][0]).nonzero()[0])  # a caption of a different vehicle
image = Image.open(gallery[j_img][1]).convert('RGB').resize((224, 224))
d_match, d_other = sampling_density(query[j][1], gallery[j_img][1]), sampling_density(query[other][1], gallery[j_img][1])

fig, axes = plt.subplots(1, 3, figsize=(13, 4.6))
for ax, d, title in [(axes[0], d_match, f'matching caption (query {j})'), (axes[1], d_other, f'other caption (query {other})')]:
    ax.imshow(image), ax.imshow(d, extent=(0, 224, 224, 0), cmap='inferno', alpha=0.6, interpolation='bilinear'), ax.set_title(title, fontsize=9)
lim = np.abs(d_match - d_other).max()
axes[2].imshow(image), axes[2].imshow(d_match - d_other, extent=(0, 224, 224, 0), cmap='bwr', vmin=-lim, vmax=lim, alpha=0.6, interpolation='bilinear')
axes[2].set_title('matching − other', fontsize=9)
for ax in axes:
    ax.set_xticks([]), ax.set_yticks([])
plt.tight_layout(), plt.show()
print('matching:', textwrap.fill(query[j][1], 150))
print('other:   ', textwrap.fill(query[other][1], 150))
print(f'total variation between the two maps: {0.5 * np.abs(d_match - d_other).sum():.3f} (0 = identical, 1 = disjoint)')